# Phase 4c — Robustness & Validity Checks

Loads **both** n=75 and n=127 tensors (needed for the disjoint follow-up evaluation and
cross-sample comparisons) and runs every additional validity check reported in the
paper: prompt-grouped cross-validation, the disjoint follow-up evaluation with a
bootstrap CI, dimensionality-reduced classification, coefficient stability across
folds, a prompt-aware permutation test for H4, a ground-truth labeling sensitivity
check, and a full audit of flagged prompts.

Independent of Phase 4a/4b — rebuilds every feature from scratch. Slower to run (H1–H4
computed twice, once per sample size) but has no dependency on prior notebook state,
which is the point: every number below can be reproduced from this notebook alone.

## Imports, paths, and shared utilities

`compute_features` is the single source of truth for the full 1,009-dimensional feature
vector (H1+H2+H3+H4) — every check below calls this rather than reimplementing feature
computation, so there's no risk of two checks silently using different H3 definitions.

In [2]:
import os
import json
from collections import Counter, defaultdict

import numpy as np

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


np.random.seed(42)

NPZ_75_PATH   = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v2.npz'
META_75_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v2.jsonl'
NPZ_127_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v3.npz'
META_127_PATH = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v3.jsonl'
LABELED_COMBINED_PATH = '/kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl'
OUTPUT_DIR = '/kaggle/working/pivot_research'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_LAYERS, N_HEADS, N_QUERIES, EARLY_KEY_RANGE = 28, 12, 30, 50
EARLY_LAYERS = list(range(0, N_LAYERS // 3))
LATE_LAYERS  = list(range(2 * N_LAYERS // 3, N_LAYERS))


def entropy_structural(row, valid_start, query_index):
    """Normalized entropy over real, causally available captured keys."""

    real = row[valid_start:].astype(np.float64)
    total_real_keys = len(real)

    K_q = total_real_keys - (
        N_QUERIES - 1 - query_index
    )
    K_q = max(1, min(K_q, total_real_keys))

    available = real[:K_q]
    mass = available.sum()

    if K_q <= 1 or mass <= 0:
        return 0.0

    probabilities = available / mass
    nonzero = probabilities > 0

    raw_entropy = -np.sum(
        probabilities[nonzero]
        * np.log(probabilities[nonzero])
    )

    return raw_entropy / np.log(K_q)


def load_metadata(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def compute_features(npz_path, metadata):
    """Builds the full 1,009-dim feature matrix (H1+H2+H3+H4) for one sample."""
    npz = np.load(npz_path)
    n = len(metadata)
    weights = np.stack([npz[f'weights_{i:03d}'] for i in range(n)], axis=0)
    outputs = np.stack([npz[f'outputs_{i:03d}'] for i in range(n)], axis=0)
    valid_key_starts = np.array([m['valid_key_start'] for m in metadata])

    h1 = np.zeros((n, N_LAYERS, N_HEADS))
    for i in range(n):
        v = valid_key_starts[i]
        early_end = min(v + EARLY_KEY_RANGE, weights.shape[-1])
        h1[i] = weights[i, :, :, :, v:early_end].sum(axis=-1).mean(axis=-1)

    h2 = np.linalg.norm(outputs, axis=-1).mean(axis=2)

    h3 = np.zeros((n, N_LAYERS, N_HEADS))
    for i in range(n):
        v = valid_key_starts[i]
        for L in range(N_LAYERS):
            for H in range(N_HEADS):
                ents = [
                        entropy_structural( weights[i, L, H, q, :], v, q, )
                        for q in range(N_QUERIES)
                        ]
                h3[i, L, H] = np.mean(ents)

    early_mag = h2[:, EARLY_LAYERS, :].mean(axis=(1, 2))
    late_mag  = h2[:, LATE_LAYERS, :].mean(axis=(1, 2))
    h4 = late_mag / (early_mag + 1e-10)

    X = np.concatenate([h1.reshape(n, -1), h2.reshape(n, -1), h3.reshape(n, -1), h4.reshape(n, 1)], axis=1)
    y = np.array([1 if m['label'] == 'correction' else 0 for m in metadata])
    prompt_ids = np.array([m['prompt_id'] for m in metadata])
    return X, y, prompt_ids


def permutation_test(g1, g2, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = g1.mean() - g2.mean()
    combined = np.concatenate([g1, g2])
    n1 = len(g1)
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        rng.shuffle(combined)
        diffs[i] = combined[:n1].mean() - combined[n1:].mean()
    return observed, (np.abs(diffs) >= np.abs(observed)).mean()

## Build all feature matrices

This is the slow cell — the entropy loop runs once per sample size. Also isolates the
52 newly-collected records (`new_mask`, prompts starting with `mis2_`) needed for the
disjoint follow-up evaluation below.

In [3]:
print("Building n=75 features...")
meta_75 = load_metadata(META_75_PATH)
X75, y75, pid75 = compute_features(NPZ_75_PATH, meta_75)

print("Building n=127 features...")
meta_127 = load_metadata(META_127_PATH)
X127, y127, pid127 = compute_features(NPZ_127_PATH, meta_127)

new_mask = np.array([m['prompt_id'].startswith('mis2_') for m in meta_127])
Xnew, ynew, pidnew = X127[new_mask], y127[new_mask], pid127[new_mask]

print(f"n=75:  {dict(Counter(y75))}")
print(f"n=127: {dict(Counter(y127))}")
print(f"new-52: {dict(Counter(ynew))}")
assert len(ynew) == 52, f"Expected 52 new records, got {len(ynew)}"

assert X75.shape == (75, 1009), X75.shape
assert len(y75) == 75
assert int(y75.sum()) == 14
assert int((y75 == 0).sum()) == 61

assert X127.shape == (127, 1009), X127.shape
assert len(y127) == 127
assert int(y127.sum()) == 22
assert int((y127 == 0).sum()) == 105

assert Xnew.shape == (52, 1009), Xnew.shape
assert int(ynew.sum()) == 8
assert int((ynew == 0).sum()) == 44

assert np.isfinite(X75).all()
assert np.isfinite(X127).all()
assert np.isfinite(Xnew).all()

Building n=75 features...
Building n=127 features...
n=75:  {np.int64(0): 61, np.int64(1): 14}
n=127: {np.int64(0): 105, np.int64(1): 22}
new-52: {np.int64(0): 44, np.int64(1): 8}


## Instance-level CV (baseline, both samples)

Reproduces Phase 4a/4b's classifier numbers as a sanity check that this notebook's
`compute_features` matches theirs exactly.

In [4]:
def instance_cv(X, y, seed=42):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    preds, probas = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in skf.split(X, y):
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[tr])
        preds[te] = clf.predict(Xte); probas[te] = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(y, probas), balanced_accuracy_score(y, preds), f1_score(y, preds, zero_division=0)

auc_75, ba_75, f1_75 = instance_cv(X75, y75)
auc_127, ba_127, f1_127 = instance_cv(X127, y127)
print(f"Instance-level CV n=75:  AUC={auc_75:.3f}, bal_acc={ba_75:.3f}, F1={f1_75:.3f}")
print(f"Instance-level CV n=127: AUC={auc_127:.3f}, bal_acc={ba_127:.3f}, F1={f1_127:.3f}")

Instance-level CV n=75:  AUC=0.720, bal_acc=0.572, F1=0.312
Instance-level CV n=127: AUC=0.579, bal_acc=0.514, F1=0.208


## Prompt-grouped cross-validation

With 15 samples generated per prompt, ordinary instance-level CV can place different
samples from the same prompt in both training and test folds — a classifier could
partly learn to recognize a familiar prompt rather than a genuine, transferable signal.
`StratifiedGroupKFold` groups by prompt so all records from one prompt stay entirely
within either train or test.

In [5]:
def grouped_cv(X, y, groups, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    preds, probas = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in sgkf.split(X, y, groups=groups):
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[tr])
        preds[te] = clf.predict(Xte); probas[te] = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(y, probas), balanced_accuracy_score(y, preds)

g_auc_75, g_ba_75 = grouped_cv(X75, y75, pid75)
g_auc_127, g_ba_127 = grouped_cv(X127, y127, pid127)
print(f"Prompt-grouped CV n=75:  AUC={g_auc_75:.3f}, bal_acc={g_ba_75:.3f}")
print(f"Prompt-grouped CV n=127: AUC={g_auc_127:.3f}, bal_acc={g_ba_127:.3f}")

Prompt-grouped CV n=75:  AUC=0.679, bal_acc=0.536
Prompt-grouped CV n=127: AUC=0.473, bal_acc=0.434


## Disjoint follow-up evaluation

The strictest test available in this dataset: train once on the original 75 records,
freeze the model, evaluate purely on the 52 records collected in the second, later
round — which the model never saw during training. Not a fully independent held-out
test in the strictest sense (the 52 records had already been examined in aggregate via
the pooled analyses above), and it's a test under domain shift (the 52 are entirely
misdirection-domain), not a same-distribution replication — see the paper's Section 3.1
for the full discussion of what this evaluation does and doesn't establish.

The bootstrap CI resamples **prompts**, not individual records, since records from the
same prompt are not independent.

In [6]:
sc = StandardScaler(); X75s, Xnews = sc.fit_transform(X75), sc.transform(Xnew)
clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
clf.fit(X75s, y75)
preds_new, probas_new = clf.predict(Xnews), clf.predict_proba(Xnews)[:, 1]

d_auc = roc_auc_score(ynew, probas_new)
d_ba = balanced_accuracy_score(ynew, preds_new)
d_f1 = f1_score(ynew, preds_new, zero_division=0)
print(f"Disjoint follow-up: AUC={d_auc:.3f}, bal_acc={d_ba:.3f}, F1={d_f1:.3f}")

rng = np.random.default_rng(42)
unique_prompts = np.array(sorted(set(pidnew)))
boot_aucs = []
for _ in range(2000):
    sampled = rng.choice(unique_prompts, size=len(unique_prompts), replace=True)
    idx = np.concatenate([np.where(pidnew == p)[0] for p in sampled])
    y_boot = ynew[idx]
    if len(set(y_boot)) < 2:
        continue
    try:
        boot_aucs.append(roc_auc_score(y_boot, probas_new[idx]))
    except ValueError:
        continue
boot_aucs = np.array(boot_aucs)
ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])
print(f"95% CI (prompt-level bootstrap, {len(boot_aucs)}/2000 valid): [{ci_low:.3f}, {ci_high:.3f}]")

Disjoint follow-up: AUC=0.392, bal_acc=0.494, F1=0.244
95% CI (prompt-level bootstrap, 1996/2000 valid): [0.179, 0.605]


## Cross-domain ablation (both samples)

Reproduces Phase 4a/4b's cross-domain numbers side by side, for the paper's main
results table.

In [7]:
print("Cross-domain ablation:")
for label, X, y, categories in [
    ('n=75', X75, y75, np.array([m['category'] for m in meta_75])),
    ('n=127', X127, y127, np.array([m['category'] for m in meta_127])),
]:
    print(f"\n{label}:")
    for held_out in ['math', 'logic', 'misdirection']:
        test_mask = categories == held_out
        train_mask = ~test_mask
        sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X[train_mask]), sc.transform(X[test_mask])
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
        clf.fit(Xtr, y[train_mask])
        preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]
        print(f"  {held_out}: bal_acc={balanced_accuracy_score(y[test_mask], preds):.3f}, "
              f"AUC={roc_auc_score(y[test_mask], probas):.3f}")

Cross-domain ablation:

n=75:
  math: bal_acc=0.576, AUC=0.707
  logic: bal_acc=0.350, AUC=0.525
  misdirection: bal_acc=0.518, AUC=0.554

n=127:
  math: bal_acc=0.318, AUC=0.222
  logic: bal_acc=0.500, AUC=0.412
  misdirection: bal_acc=0.488, AUC=0.421


## Appendix D.2 — Dimensionality-reduced classification

Re-runs the classifier under PCA and univariate feature selection (SelectKBest), using
`Pipeline` so feature reduction is refit inside each CV training fold (avoiding
leakage from selecting features on the full dataset before splitting). Also computes
the best single-feature univariate AUC — reported descriptively only, since selecting
the max of 1,009 comparisons is strongly selection-biased.

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'balanced_accuracy': 'balanced_accuracy', 'roc_auc': 'roc_auc'}

print("D.2: Dimensionality reduction")
for label, X, y, pca_comps in [('n=75', X75, y75, [5, 10, 20, 30, 45]), ('n=127', X127, y127, [5, 10, 20, 50, 100])]:
    print(f"\n{label}:")
    for n_comp in pca_comps:
        pipe = Pipeline([('scale', StandardScaler()), ('pca', PCA(n_components=n_comp, random_state=42)),
                          ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, C=0.1, random_state=42))])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        print(f"  PCA-{n_comp}: bal_acc={scores['test_balanced_accuracy'].mean():.3f}±{scores['test_balanced_accuracy'].std():.3f}, "
              f"AUC={scores['test_roc_auc'].mean():.3f}±{scores['test_roc_auc'].std():.3f}")
    for k in [5, 10, 20, 50, 100]:
        pipe = Pipeline([('scale', StandardScaler()), ('select', SelectKBest(score_func=f_classif, k=k)),
                          ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, C=0.1, random_state=42))])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
        print(f"  SelectKBest-{k}: bal_acc={scores['test_balanced_accuracy'].mean():.3f}±{scores['test_balanced_accuracy'].std():.3f}, "
              f"AUC={scores['test_roc_auc'].mean():.3f}±{scores['test_roc_auc'].std():.3f}")
    single_aucs = []
    for i in range(X.shape[1]):
        try:
            auc = roc_auc_score(y, X[:, i]); single_aucs.append(max(auc, 1 - auc))
        except ValueError:
            continue
    single_aucs = np.array(single_aucs)
    print(f"  Best single-feature AUC: {single_aucs.max():.3f}, mean: {single_aucs.mean():.3f}")

D.2: Dimensionality reduction

n=75:
  PCA-5: bal_acc=0.662±0.126, AUC=0.695±0.054
  PCA-10: bal_acc=0.635±0.180, AUC=0.772±0.089
  PCA-20: bal_acc=0.502±0.171, AUC=0.658±0.123
  PCA-30: bal_acc=0.601±0.162, AUC=0.767±0.109
  PCA-45: bal_acc=0.610±0.156, AUC=0.762±0.094
  SelectKBest-5: bal_acc=0.671±0.083, AUC=0.721±0.085
  SelectKBest-10: bal_acc=0.595±0.087, AUC=0.711±0.034
  SelectKBest-20: bal_acc=0.570±0.145, AUC=0.694±0.068
  SelectKBest-50: bal_acc=0.619±0.192, AUC=0.727±0.069
  SelectKBest-100: bal_acc=0.577±0.128, AUC=0.715±0.046
  Best single-feature AUC: 0.804, mean: 0.593

n=127:
  PCA-5: bal_acc=0.457±0.138, AUC=0.501±0.138
  PCA-10: bal_acc=0.462±0.108, AUC=0.494±0.091
  PCA-20: bal_acc=0.468±0.130, AUC=0.463±0.117
  PCA-50: bal_acc=0.515±0.084, AUC=0.571±0.123
  PCA-100: bal_acc=0.528±0.099, AUC=0.586±0.121
  SelectKBest-5: bal_acc=0.539±0.100, AUC=0.576±0.093
  SelectKBest-10: bal_acc=0.478±0.145, AUC=0.581±0.102
  SelectKBest-20: bal_acc=0.507±0.151, AUC=0.566±0.070
 

## Appendix D.3 — Coefficient stability across folds (n=75)

Are the classifier's learned feature weights stable across CV folds? Computes pairwise
correlation between folds' coefficient vectors and the fraction of features with
unanimous sign agreement, then identifies which features are both high-magnitude and
stable, mapped back to their originating hypothesis block. See the paper for why high
stability here is *not* strong evidence of a generalizable signal (folds share 75% of
their training data by construction).

In [9]:
print("D.3: Coefficient stability, n=75")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_coefs = []
for tr, te in skf.split(X75, y75):
    sc = StandardScaler(); Xtr = sc.fit_transform(X75[tr])
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
    clf.fit(Xtr, y75[tr])
    fold_coefs.append(clf.coef_[0].copy())
fold_coefs = np.array(fold_coefs)

corr_matrix = np.corrcoef(fold_coefs)
mean_corr = np.mean([corr_matrix[i, j] for i in range(5) for j in range(i + 1, 5)])
print(f"Mean pairwise correlation: {mean_corr:.3f}")

signs = np.sign(fold_coefs)
unanimous = np.all(signs == signs[0], axis=0)
print(f"Unanimous sign agreement: {unanimous.sum()} / 1009 ({unanimous.mean():.1%})")

mean_abs_coef = np.abs(fold_coefs).mean(axis=0)
stable_and_strong = unanimous & (mean_abs_coef > np.percentile(mean_abs_coef, 95))
idx = np.where(stable_and_strong)[0]
h1_count = sum(1 for i in idx if i < 336)
h2_count = sum(1 for i in idx if 336 <= i < 672)
h3_count = sum(1 for i in idx if 672 <= i < 1008)
h4_count = sum(1 for i in idx if i == 1008)
print(f"Stable+strong: {len(idx)} total -- H1:{h1_count}, H2:{h2_count}, H3:{h3_count}, H4:{h4_count}")

D.3: Coefficient stability, n=75
Mean pairwise correlation: 0.795
Unanimous sign agreement: 611 / 1009 (60.6%)
Stable+strong: 49 total -- H1:0, H2:44, H3:5, H4:0


## H4 prompt-aware permutation test

Tests whether H4's record-level p-value (closest to significance of the four
hypotheses) is sensitive to prompt clustering: labels are shuffled only *within* each
prompt's own records, preserving each prompt's exact class composition, rather than
freely across the full sample. Only prompts containing both classes contribute
variability under this scheme — printed below as a diagnostic, since this substantially
reduces effective power relative to the record-level test.

In [10]:
def within_prompt_permute(y, prompt_ids, rng):
    y_perm = y.copy()
    groups = defaultdict(list)
    for i, pid in enumerate(prompt_ids):
        groups[pid].append(i)
    for pid, idx in groups.items():
        if len(idx) > 1:
            idx = np.array(idx)
            y_perm[idx] = rng.permutation(y[idx])
    return y_perm

def prompt_aware_permutation_test(feature_values, y, prompt_ids, n_perm=10000, seed=42):
    rng = np.random.default_rng(seed)
    prompt_ids = np.array(prompt_ids)
    observed_diff = feature_values[y == 1].mean() - feature_values[y == 0].mean()
    mixed = sum(1 for pid in set(prompt_ids) if len(set(y[prompt_ids == pid])) > 1)
    null_diffs = np.zeros(n_perm)
    for i in range(n_perm):
        y_perm = within_prompt_permute(y, prompt_ids, rng)
        null_diffs[i] = feature_values[y_perm == 1].mean() - feature_values[y_perm == 0].mean()
    p_value = (np.abs(null_diffs) >= np.abs(observed_diff)).mean()
    return observed_diff, p_value, mixed

print("H4 prompt-aware permutation test:")
h4_75, h4_127 = X75[:, -1], X127[:, -1]
obs_75, p_75, mixed_75 = prompt_aware_permutation_test(h4_75, y75, pid75)
print(f"  n=75:  diff={obs_75:.4f}, p={p_75:.4f} ({mixed_75} mixed-class prompts of {len(set(pid75))})")
obs_127, p_127, mixed_127 = prompt_aware_permutation_test(h4_127, y127, pid127)
print(f"  n=127: diff={obs_127:.4f}, p={p_127:.4f} ({mixed_127} mixed-class prompts of {len(set(pid127))})")

H4 prompt-aware permutation test:
  n=75:  diff=-0.0798, p=0.0842 (6 mixed-class prompts of 39)
  n=127: diff=-0.0336, p=0.4420 (10 mixed-class prompts of 55)


## Ground-truth labeling sensitivity check

Does the headline finding survive if the second reviewer's alternative labels are used
for the 5 disputed correction/hallucination_pivot records (Appendix C)? Rebuilds
features under the alternative labeling and re-runs the two central evaluations.

In [11]:
REVIEWER_ALTERNATIVE_LABELS = {
    ('mis2_04', 2):  'hallucination_pivot',
    ('mis2_16', 0):  'hallucination_pivot',
    ('math_27', 2):  'correction',
    ('logic_20', 7): 'correction',
    ('math_23', 8):  'correction',
}

def apply_reviewer_labels(metadata):
    meta_alt = []
    n_changed = 0
    for m in metadata:
        m2 = dict(m)
        key = (m['prompt_id'], m['sample_idx'])
        if key in REVIEWER_ALTERNATIVE_LABELS:
            m2['label'] = REVIEWER_ALTERNATIVE_LABELS[key]
            n_changed += 1
        meta_alt.append(m2)
    return meta_alt, n_changed

print("Ground-truth sensitivity check (alternative reviewer labels):")
meta_75_alt, n_75_changed = apply_reviewer_labels(meta_75)
X75_alt, y75_alt, _ = compute_features(NPZ_75_PATH, meta_75_alt)
print(f"  n=75: {n_75_changed} labels changed")

meta_127_alt, n_127_changed = apply_reviewer_labels(meta_127)
X127_alt, y127_alt, _ = compute_features(NPZ_127_PATH, meta_127_alt)
new_mask_alt = np.array([m['prompt_id'].startswith('mis2_') for m in meta_127_alt])
Xnew_alt, ynew_alt = X127_alt[new_mask_alt], y127_alt[new_mask_alt]
print(f"  n=127: {n_127_changed} labels changed")

auc_127_alt, ba_127_alt, _ = instance_cv(X127_alt, y127_alt)
print(f"  Pooled n=127 CV, alt labels: AUC={auc_127_alt:.3f} (original: {auc_127:.3f})")

sc = StandardScaler(); X75_alt_s, Xnew_alt_s = sc.fit_transform(X75_alt), sc.transform(Xnew_alt)
clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)
clf.fit(X75_alt_s, y75_alt)
probas_new_alt = clf.predict_proba(Xnew_alt_s)[:, 1]
d_auc_alt = roc_auc_score(ynew_alt, probas_new_alt)
print(f"  Disjoint follow-up, alt labels: AUC={d_auc_alt:.3f} (original: {d_auc:.3f})")

Ground-truth sensitivity check (alternative reviewer labels):
  n=75: 2 labels changed
  n=127: 4 labels changed
  Pooled n=127 CV, alt labels: AUC=0.449 (original: 0.579)
  Disjoint follow-up, alt labels: AUC=0.467 (original: 0.392)


## Full 9-prompt ground-truth audit scope check

Checks how many usable records trace to each of the nine prompts flagged during the
full ground-truth audit (Limitations) — establishes whether any of them could
plausibly be driving the paper's numbers.

In [12]:
FLAGGED_PROMPTS = ['logic_17', 'logic_03', 'mis_05', 'mis2_06', 'mis_08', 'math_13', 'logic_02', 'logic_14', 'math_36']

with open(LABELED_COMBINED_PATH) as f:
    all_records = [json.loads(l) for l in f if l.strip()]

print(f"Ground-truth audit scope check ({len(all_records)} total records):")
total_flagged_usable = 0
for pid in FLAGGED_PROMPTS:
    matches = [r for r in all_records if r['prompt_id'] == pid]
    usable = [r for r in matches if r['label'] in ('correction', 'hallucination_pivot') and r['pivot_token_start'] >= 30]
    total_flagged_usable += len(usable)
    print(f"  {pid}: {len(matches)} total, {len(usable)} usable")

n_overall_usable = sum(1 for r in all_records if r['label'] in ('correction', 'hallucination_pivot') and r['pivot_token_start'] >= 30)
print(f"\nTotal usable from flagged prompts: {total_flagged_usable} / {n_overall_usable}")

Ground-truth audit scope check (155 total records):
  logic_17: 0 total, 0 usable
  logic_03: 1 total, 1 usable
  mis_05: 0 total, 0 usable
  mis2_06: 3 total, 1 usable
  mis_08: 5 total, 2 usable
  math_13: 0 total, 0 usable
  logic_02: 1 total, 1 usable
  logic_14: 0 total, 0 usable
  math_36: 0 total, 0 usable

Total usable from flagged prompts: 5 / 127


## Final label-count verification

Ground-truth check on the overall and per-batch label distribution — the single source
of truth for every count reported in the paper's provenance table (Section 3.2).

In [13]:
overall = Counter(r['label'] for r in all_records)
batch1 = [r for r in all_records if not r['prompt_id'].startswith('mis2_')]
batch2 = [r for r in all_records if r['prompt_id'].startswith('mis2_')]
print(f"Final count verification:")
print(f"  Overall: {dict(overall)}")
print(f"  Batch 1 ({len(batch1)}): {dict(Counter(r['label'] for r in batch1))}")
print(f"  Batch 2 ({len(batch2)}): {dict(Counter(r['label'] for r in batch2))}")
print(f"  Paper claims: 23 correction / 110 hallucination_pivot / 22 uncertain")

Final count verification:
  Overall: {'hallucination_pivot': 110, 'uncertain': 22, 'correction': 23}
  Batch 1 (93): {'hallucination_pivot': 65, 'uncertain': 14, 'correction': 14}
  Batch 2 (62): {'hallucination_pivot': 45, 'uncertain': 8, 'correction': 9}
  Paper claims: 23 correction / 110 hallucination_pivot / 22 uncertain
